# Setup - Imports and Configuration

In [ ]:
import sys
import os
import json
import logging
import time
import threading
from typing import Dict, Optional, Tuple
from datetime import datetime, timedelta
from pathlib import Path

import numpy as np
import pandas as pd
import yfinance as yf
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from catboost import CatBoostRegressor
from fastapi import FastAPI, HTTPException, Request
from fastapi.middleware.cors import CORSMiddleware
from fastapi.middleware.gzip import GZipMiddleware
from fastapi.responses import JSONResponse
from pydantic import BaseModel, Field
import joblib
import uvicorn
import nest_asyncio

# Apply nest_asyncio for notebook compatibility
nest_asyncio.apply()

print("✅ All imports loaded")

# ============================================
# Configuration
# ============================================

class Config:
    """Application configuration"""
    PROJECT_ROOT = Path.cwd()
    MODELS_PATH = PROJECT_ROOT / 'models'
    LOGS_PATH = PROJECT_ROOT / 'logs'
    CACHE_PATH = PROJECT_ROOT / 'cache'
    
    MODEL_FILENAME = 'production_model.pkl'
    MODEL_VERSION_FILENAME = 'model_version.json'
    
    API_TITLE = "S&P 500 Predictor API"
    API_VERSION = "2.0.0"
    API_DESCRIPTION = "Machine Learning Powered S&P 500 Predictor"
    
    RATE_LIMIT = "10/minute"
    LOG_LEVEL = logging.INFO
    PREDICTION_CACHE_TTL = 300
    
    @classmethod
    def ensure_directories(cls):
        for path in [cls.MODELS_PATH, cls.LOGS_PATH, cls.CACHE_PATH]:
            path.mkdir(parents=True, exist_ok=True)

Config.ensure_directories()

print("✅ Configuration loaded")
print(f"📁 Models: {Config.MODELS_PATH}")
print(f"📁 Logs: {Config.LOGS_PATH}")
print(f"📁 Cache: {Config.CACHE_PATH}")

# ============================================
# Logging Setup
# ============================================

def setup_logging():
    log_format = "%(asctime)s - %(name)s - %(levelname)s - %(message)s"
    logging.basicConfig(
        level=Config.LOG_LEVEL,
        format=log_format,
        handlers=[
            logging.FileHandler(Config.LOGS_PATH / 'api.log'),
            logging.StreamHandler()
        ]
    )
    return logging.getLogger(__name__)

logger = setup_logging()
print("✅ Logging configured")

✅ All imports loaded
✅ Configuration loaded
📁 Models: c:\Users\nyvra\Downloads\sp500-predictor\models
📁 Logs: c:\Users\nyvra\Downloads\sp500-predictor\logs
📁 Cache: c:\Users\nyvra\Downloads\sp500-predictor\cache
✅ Logging configured


# Cache Manager

In [ ]:
class CacheManager:
    def __init__(self):
        self._cache = {}
    
    def get(self, key: str):
        if key in self._cache:
            value, expiry = self._cache[key]
            if datetime.now() < expiry:
                return value
            del self._cache[key]
        return None
    
    def set(self, key: str, value: any, ttl_seconds: int = 300):
        expiry = datetime.now() + timedelta(seconds=ttl_seconds)
        self._cache[key] = (value, expiry)
    
    def clear(self):
        self._cache.clear()

cache = CacheManager()
print("✅ Cache manager initialized")

✅ Cache manager initialized


# Data Collector

In [ ]:
class SimpleDataCollector:
    @staticmethod
    def fetch_data(start_date='2010-01-01', end_date=None, max_retries=3):
        if end_date is None:
            end_date = datetime.now().strftime('%Y-%m-%d')
        
        for attempt in range(max_retries):
            try:
                logger.info(f"Fetching data (attempt {attempt + 1})")
                ticker = yf.Ticker("^GSPC")
                df = ticker.history(start=start_date, end=end_date)
                
                if df.empty:
                    raise ValueError("No data returned")
                
                df.columns = [col.lower() for col in df.columns]
                df['returns'] = df['close'].pct_change()
                df['target'] = df['close'].shift(-5) / df['close'] - 1
                df = df.dropna()
                
                logger.info(f"✅ Fetched {len(df)} rows")
                return df
                
            except Exception as e:
                logger.warning(f"Attempt {attempt + 1} failed: {e}")
                if attempt == max_retries - 1:
                    raise
                time.sleep(2 ** attempt)
        
        raise RuntimeError("Failed to fetch data")
    
    @staticmethod
    def create_features(df):
        df['sma_20'] = df['close'].rolling(20).mean()
        df['sma_50'] = df['close'].rolling(50).mean()
        df['volatility'] = df['returns'].rolling(20).std() * np.sqrt(252)
        df['close_vs_sma20'] = (df['close'] - df['sma_20']) / df['sma_20']
        df['close_vs_sma50'] = (df['close'] - df['sma_50']) / df['sma_50']
        df['volume_ratio'] = df['volume'] / df['volume'].rolling(20).mean()
        df = df.dropna()
        
        feature_cols = ['open', 'high', 'low', 'close', 'volume', 'returns',
                       'volatility', 'close_vs_sma20', 'close_vs_sma50', 'volume_ratio']
        X = df[feature_cols].values
        y = df['target'].values
        
        return X, y, feature_cols

print("✅ Data collector ready")

✅ Data collector ready


# Training Pipeline

In [ ]:
class SimpleTrainingPipeline:
    def __init__(self):
        self.imputer = SimpleImputer(strategy='mean')
        self.scaler = StandardScaler()
        self.model = None
        self.feature_names = None
    
    def train(self, retrain_reason='manual'):
        logger.info(f"🚀 Starting training (Reason: {retrain_reason})")
        start_time = time.time()
        
        df = SimpleDataCollector.fetch_data()
        X, y, feature_names = SimpleDataCollector.create_features(df)
        self.feature_names = feature_names
        
        split_idx = int(len(X) * 0.8)
        X_train, X_test = X[:split_idx], X[split_idx:]
        y_train, y_test = y[:split_idx], y[split_idx:]
        
        X_train = self.imputer.fit_transform(X_train)
        X_test = self.imputer.transform(X_test)
        X_train = self.scaler.fit_transform(X_train)
        X_test = self.scaler.transform(X_test)
        
        self.model = CatBoostRegressor(
            iterations=500, depth=6, learning_rate=0.1,
            random_seed=42, verbose=False, early_stopping_rounds=50
        )
        self.model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
        
        y_pred = self.model.predict(X_test)
        rmse = np.sqrt(((y_test - y_pred) ** 2).mean())
        mae = np.mean(np.abs(y_test - y_pred))
        r2 = 1 - (np.sum((y_test - y_pred) ** 2) / np.sum((y_test - np.mean(y_test)) ** 2))
        
        training_time = time.time() - start_time
        
        artifacts = {
            'model': self.model,
            'imputer': self.imputer,
            'scaler': self.scaler,
            'feature_names': self.feature_names,
            'metrics': {
                'test_rmse': float(rmse),
                'test_mae': float(mae),
                'test_r2': float(r2),
                'training_time': training_time,
                'n_samples': len(y),
                'n_features': len(feature_names)
            },
            'training_date': datetime.now().isoformat(),
            'retrain_reason': retrain_reason,
            'model_version': Config.API_VERSION
        }
        
        joblib.dump(artifacts, Config.MODELS_PATH / Config.MODEL_FILENAME)
        
        version_path = Config.MODELS_PATH / Config.MODEL_VERSION_FILENAME
        with open(version_path, 'w') as f:
            json.dump({
                'version': Config.API_VERSION,
                'training_date': artifacts['training_date'],
                'metrics': artifacts['metrics']
            }, f, indent=2)
        
        logger.info(f"✅ Training complete! RMSE: {rmse:.4f}")
        return artifacts
    
    def load_model(self):
        model_path = Config.MODELS_PATH / Config.MODEL_FILENAME
        if not model_path.exists():
            return self.train(retrain_reason='initial')
        
        artifacts = joblib.load(model_path)
        self.model = artifacts['model']
        self.imputer = artifacts['imputer']
        self.scaler = artifacts['scaler']
        self.feature_names = artifacts['feature_names']
        
        logger.info(f"✅ Model loaded (RMSE: {artifacts['metrics']['test_rmse']:.4f})")
        return artifacts

print("✅ Training pipeline ready")

✅ Training pipeline ready


# Prediction Pipeline

In [ ]:
class SimplePredictionPipeline:
    def __init__(self):
        self.trainer = SimpleTrainingPipeline()
    
    def predict(self, use_cache=True):
        cache_key = "latest_prediction"
        
        if use_cache:
            cached = cache.get(cache_key)
            if cached:
                logger.info("Returning cached prediction")
                return cached
        
        artifacts = self.trainer.load_model()
        
        ticker = yf.Ticker("^GSPC")
        df = ticker.history(period="60d")
        df.columns = [col.lower() for col in df.columns]
        df['returns'] = df['close'].pct_change()
        df['sma_20'] = df['close'].rolling(20).mean()
        df['sma_50'] = df['close'].rolling(50).mean()
        df['volatility'] = df['returns'].rolling(20).std() * np.sqrt(252)
        df['close_vs_sma20'] = (df['close'] - df['sma_20']) / df['sma_20']
        df['close_vs_sma50'] = (df['close'] - df['sma_50']) / df['sma_50']
        df['volume_ratio'] = df['volume'] / df['volume'].rolling(20).mean()
        
        latest = df.iloc[-1]
        features = np.array([[
            latest['open'], latest['high'], latest['low'], latest['close'],
            latest['volume'], latest['returns'], latest['volatility'],
            latest['close_vs_sma20'], latest['close_vs_sma50'], latest['volume_ratio']
        ]])
        
        features = artifacts['imputer'].transform(features)
        features = artifacts['scaler'].transform(features)
        prediction = artifacts['model'].predict(features)[0]
        
        abs_pred = abs(prediction)
        if abs_pred > 0.02:
            confidence = "High"
            recommendation = "Strong " + ("BUY" if prediction > 0 else "SELL")
        elif abs_pred > 0.01:
            confidence = "Medium"
            recommendation = "Cautious " + ("BUY" if prediction > 0 else "SELL")
        else:
            confidence = "Low"
            recommendation = "HOLD"
        
        result = {
            'prediction': float(prediction),
            'prediction_percent': f"{prediction:.4%}",
            'direction': 'BULLISH' if prediction > 0 else 'BEARISH',
            'confidence': confidence,
            'recommendation': recommendation,
            'current_price': float(latest['close']),
            'timestamp': datetime.now().isoformat(),
            'model_version': Config.API_VERSION
        }
        
        log_entry = {
            'timestamp': result['timestamp'],
            'prediction': result['prediction'],
            'direction': result['direction']
        }
        
        with open(Config.LOGS_PATH / 'predictions.log', 'a') as f:
            f.write(json.dumps(log_entry) + '\n')
        
        cache.set(cache_key, result, Config.PREDICTION_CACHE_TTL)
        return result

print("✅ Prediction pipeline ready")

✅ Prediction pipeline ready


# Orchestrator

In [ ]:
class SimpleOrchestrator:
    def __init__(self):
        self.trainer = SimpleTrainingPipeline()
        self.predictor = SimplePredictionPipeline()
    
    def run_full_training(self, retrain_reason='manual'):
        return self.trainer.train(retrain_reason=retrain_reason)
    
    def run_prediction(self):
        return self.predictor.predict()

print("✅ Orchestrator ready")

✅ Orchestrator ready


# Pydantic Models

In [ ]:
from typing import Optional

class PredictionResponse(BaseModel):
    prediction: float
    prediction_percent: str
    direction: str
    confidence: str
    recommendation: str
    current_price: float
    timestamp: str
    model_version: str

class TrainResponse(BaseModel):
    status: str
    rmse: float
    mae: float
    r2: float
    training_date: str
    message: str

class HealthResponse(BaseModel):
    status: str
    model_loaded: bool
    model_exists: bool
    model_version: str
    last_training_date: Optional[str]
    last_rmse: Optional[float]
    uptime_seconds: float
    timestamp: str

class MetricsResponse(BaseModel):
    model_version: str
    training_date: str
    rmse: float
    mae: float
    r2: float
    n_samples: int
    n_features: int

print("✅ Pydantic models ready")

✅ Pydantic models ready


# FastAPI Application

In [ ]:
app = FastAPI(
    title=Config.API_TITLE,
    description=Config.API_DESCRIPTION,
    version=Config.API_VERSION,
    docs_url="/docs",
    redoc_url="/redoc"
)

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)
app.add_middleware(GZipMiddleware, minimum_size=1000)

pipeline = None

@app.on_event("startup")
async def startup_event():
    global pipeline
    try:
        pipeline = SimpleOrchestrator()
        pipeline.trainer.load_model()
        logger.info("✅ Pipeline initialized")
    except Exception as e:
        logger.error(f"Pipeline init failed: {e}")
        pipeline = None

print("✅ FastAPI app created")

✅ FastAPI app created


C:\Users\nyvra\AppData\Local\Temp\ipykernel_10932\1644861054.py:21: DeprecationWarning: 
        on_event is deprecated, use lifespan event handlers instead.

        Read more about it in the
        [FastAPI docs for Lifespan Events](https://fastapi.tiangolo.com/advanced/events/).
        
  @app.on_event("startup")


In [48]:
# Replace the /metrics endpoint in Cell 9 with this fixed version:

@app.get("/metrics", response_model=MetricsResponse, tags=["Metrics"])
async def get_metrics():
    model_path = Config.MODELS_PATH / Config.MODEL_FILENAME
    if not model_path.exists():
        raise HTTPException(status_code=404, detail="No model found")
    
    artifacts = joblib.load(model_path)
    metrics = artifacts['metrics']
    
    # Safe access with fallbacks
    return MetricsResponse(
        model_version=Config.API_VERSION,
        training_date=artifacts.get('training_date', datetime.now().isoformat()),
        rmse=metrics.get('test_rmse', metrics.get('rmse', 0.0)),
        mae=metrics.get('test_mae', metrics.get('mae', metrics.get('test_rmse', 0.0))),
        r2=metrics.get('test_r2', metrics.get('r2', -1.0)),
        n_samples=metrics.get('n_samples', 0),
        n_features=metrics.get('n_features', len(artifacts.get('feature_names', [])))
    )

# Error Handlers

In [ ]:
@app.exception_handler(HTTPException)
async def http_exception_handler(request: Request, exc: HTTPException):
    return JSONResponse(
        status_code=exc.status_code,
        content={
            "error": exc.detail,
            "status_code": exc.status_code,
            "timestamp": datetime.now().isoformat()
        }
    )

@app.exception_handler(Exception)
async def generic_exception_handler(request: Request, exc: Exception):
    logger.error(f"Unhandled exception: {exc}")
    return JSONResponse(
        status_code=500,
        content={
            "error": "Internal server error",
            "timestamp": datetime.now().isoformat()
        }
    )

print("✅ Error handlers configured")

✅ Error handlers configured


# Run API in background thread

In [ ]:
import threading

# Set start time for uptime tracking
app.state.start_time = time.time()

def run_api():
    """Run API in a separate thread"""
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="info", access_log=False)

print("\n" + "="*60)
print(f"🚀 {Config.API_TITLE}")
print("="*60)
print(f"📍 API: http://localhost:8000")
print(f"📍 Docs: http://localhost:8000/docs")
print(f"📍 Health: http://localhost:8000/health")
print("\n✅ API is running in background thread")
print("⚠️  To stop: Restart the kernel (Kernel → Restart)")
print("="*60 + "\n")

# Start API in background thread
api_thread = threading.Thread(target=run_api, daemon=True)
api_thread.start()

# Wait a moment for API to initialize
time.sleep(3)
print("✅ API ready! You can now make requests.")

INFO:     Started server process [10932]
INFO:     Waiting for application startup.
2026-04-10 23:10:26,875 - __main__ - INFO - ✅ Model loaded (RMSE: 0.0265)
2026-04-10 23:10:26,876 - __main__ - INFO - ✅ Pipeline initialized
INFO:     Application startup complete.



🚀 S&P 500 Predictor API
📍 API: http://localhost:8000
📍 Docs: http://localhost:8000/docs
📍 Health: http://localhost:8000/health

✅ API is running in background thread
⚠️  To stop: Restart the kernel (Kernel → Restart)



ERROR:    [Errno 10048] error while attempting to bind on address ('0.0.0.0', 8000): only one usage of each socket address (protocol/network address/port) is normally permitted
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.


✅ API ready! You can now make requests.


# Test API

In [ ]:
import requests

def test_api():
    base_url = "http://localhost:8000"
    
    print("Testing API...")
    print("="*40)
    
    try:
        # Test health
        response = requests.get(f"{base_url}/health", timeout=5)
        print(f"✅ Health check: {response.status_code}")
        
        if response.status_code == 200:
            data = response.json()
            print(f"   Model loaded: {data['model_loaded']}")
            print(f"   Model exists: {data['model_exists']}")
        
        # Test prediction
        response = requests.get(f"{base_url}/predict", timeout=10)
        print(f"\n✅ Prediction: {response.status_code}")
        
        if response.status_code == 200:
            data = response.json()
            print(f"   Direction: {data['direction']}")
            print(f"   Prediction: {data['prediction_percent']}")
            print(f"   Confidence: {data['confidence']}")
            print(f"   Recommendation: {data['recommendation']}")
            print(f"   Current Price: ${data['current_price']:,.2f}")
        
    except requests.exceptions.ConnectionError:
        print("❌ Cannot connect to API. Make sure Cell 11 is running.")
    except Exception as e:
        print(f"❌ Error: {e}")

# Run test
test_api()

Testing API...
✅ Health check: 200
   Model loaded: True
   Model exists: True


2026-04-10 23:10:33,972 - __main__ - INFO - ✅ Model loaded (RMSE: 0.0265)



✅ Prediction: 200
   Direction: BULLISH
   Prediction: 0.3949%
   Confidence: Low
   Recommendation: HOLD
   Current Price: $6,813.34


# Quick Prediction

In [ ]:
import requests

try:
    response = requests.get("http://localhost:8000/predict", timeout=10)
    if response.status_code == 200:
        data = response.json()
        
        print("\n" + "="*50)
        print("🔮 S&P 500 PREDICTION")
        print("="*50)
        print(f"📈 Direction: {data['direction']}")
        print(f"📊 Expected Return: {data['prediction_percent']}")
        print(f"🎯 Confidence: {data['confidence']}")
        print(f"💡 Recommendation: {data['recommendation']}")
        print(f"💰 Current Price: ${data['current_price']:,.2f}")
        print(f"⏰ Timestamp: {data['timestamp'][:19]}")
        print("="*50)
    else:
        print(f"Error: {response.status_code}")
except Exception as e:
    print(f"API not running: {e}")

2026-04-10 23:10:37,302 - __main__ - INFO - Returning cached prediction



🔮 S&P 500 PREDICTION
📈 Direction: BULLISH
📊 Expected Return: 0.3949%
🎯 Confidence: Low
💡 Recommendation: HOLD
💰 Current Price: $6,813.34
⏰ Timestamp: 2026-04-10T23:10:35


# Test API

In [ ]:
import requests
import time

# Wait a moment for API to start
time.sleep(2)

def test_api():
    base_url = "http://localhost:8000"
    
    print("Testing API...")
    print("="*40)
    
    try:
        # Test health
        response = requests.get(f"{base_url}/health", timeout=5)
        print(f"✅ Health check: {response.status_code}")
        
        if response.status_code == 200:
            data = response.json()
            print(f"   Model loaded: {data['model_loaded']}")
            print(f"   Model exists: {data['model_exists']}")
        
        # Test prediction
        response = requests.get(f"{base_url}/predict", timeout=10)
        print(f"\n✅ Prediction: {response.status_code}")
        
        if response.status_code == 200:
            data = response.json()
            print(f"   Direction: {data['direction']}")
            print(f"   Prediction: {data['prediction_percent']}")
            print(f"   Confidence: {data['confidence']}")
            print(f"   Recommendation: {data['recommendation']}")
        
        # Test metrics
        response = requests.get(f"{base_url}/metrics", timeout=5)
        print(f"\n✅ Metrics: {response.status_code}")
        
        if response.status_code == 200:
            data = response.json()
            print(f"   RMSE: {data['rmse']:.4f}")
            print(f"   R²: {data['r2']:.4f}")
            print(f"   Samples: {data['n_samples']}")
        
    except requests.exceptions.ConnectionError:
        print("❌ Cannot connect to API. Make sure Cell 13 is running.")
    except Exception as e:
        print(f"❌ Error: {e}")

# Run test
test_api()

Testing API...
✅ Health check: 200
   Model loaded: True
   Model exists: True


2026-04-10 23:10:43,449 - __main__ - INFO - Returning cached prediction



✅ Prediction: 200
   Direction: BULLISH
   Prediction: 0.3949%
   Confidence: Low
   Recommendation: HOLD


2026-04-10 23:10:45,499 - __main__ - ERROR - Unhandled exception: 'test_mae'
ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "c:\Users\nyvra\miniconda3\envs\data_science\lib\site-packages\uvicorn\protocols\http\httptools_impl.py", line 420, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
  File "c:\Users\nyvra\miniconda3\envs\data_science\lib\site-packages\uvicorn\middleware\proxy_headers.py", line 60, in __call__
    return await self.app(scope, receive, send)
  File "c:\Users\nyvra\miniconda3\envs\data_science\lib\site-packages\fastapi\applications.py", line 1163, in __call__
    await super().__call__(scope, receive, send)
  File "c:\Users\nyvra\miniconda3\envs\data_science\lib\site-packages\starlette\applications.py", line 90, in __call__
    await self.middleware_stack(scope, receive, send)
  File "c:\Users\nyvra\miniconda3\envs\data_science\lib\site-packages\starlette\middleware\errors.py", line 186, in __call__
    raise 


✅ Metrics: 500


# Fix Metrics Endpoint

In [ ]:
@app.get("/metrics", response_model=MetricsResponse, tags=["Metrics"])
async def get_metrics():
    model_path = Config.MODELS_PATH / Config.MODEL_FILENAME
    if not model_path.exists():
        raise HTTPException(status_code=404, detail="No model found")
    
    artifacts = joblib.load(model_path)
    
    # Get metrics safely
    metrics = artifacts['metrics']
    
    return MetricsResponse(
        model_version=Config.API_VERSION,
        training_date=artifacts['training_date'],
        rmse=metrics.get('test_rmse', 0.0),
        mae=metrics.get('test_mae', metrics.get('mae', 0.0)),  # Fallback
        r2=metrics.get('test_r2', metrics.get('r2', 0.0)),
        n_samples=metrics.get('n_samples', 0),
        n_features=metrics.get('n_features', 0)
    )

# Quick Prediction (Run anytime API is running)

In [ ]:
import requests

try:
    response = requests.get("http://localhost:8000/predict", timeout=10)
    if response.status_code == 200:
        data = response.json()
        
        print("\n" + "="*50)
        print("🔮 S&P 500 PREDICTION")
        print("="*50)
        print(f"📈 Direction: {data['direction']}")
        print(f"📊 Expected Return: {data['prediction_percent']}")
        print(f"🎯 Confidence: {data['confidence']}")
        print(f"💡 Recommendation: {data['recommendation']}")
        print(f"💰 Current Price: ${data['current_price']:,.2f}")
        print(f"⏰ Timestamp: {data['timestamp'][:19]}")
        print("="*50)
    else:
        print(f"Error: {response.status_code}")
except Exception as e:
    print(f"API not running: {e}")

2026-04-10 23:10:47,620 - __main__ - INFO - Returning cached prediction



🔮 S&P 500 PREDICTION
📈 Direction: BULLISH
📊 Expected Return: 0.3949%
🎯 Confidence: Low
💡 Recommendation: HOLD
💰 Current Price: $6,813.34
⏰ Timestamp: 2026-04-10T23:10:35


# Stop the API (if running in background)

In [ ]:
# Note: This may not work for daemon threads
# Restart kernel to fully stop

print("To stop the API:")
print("1. Click the ⏸️ Stop button in Jupyter toolbar")
print("2. Or restart the kernel: Kernel → Restart")
print("\nAPI will stop when kernel is interrupted")

To stop the API:
1. Click the ⏸️ Stop button in Jupyter toolbar
2. Or restart the kernel: Kernel → Restart

API will stop when kernel is interrupted


In [58]:
# Cell: Prometheus Metrics Setup (Run once)
from prometheus_client import Counter, Histogram, generate_latest, CONTENT_TYPE_LATEST, REGISTRY
from fastapi import Response

# Clear existing metrics to avoid duplicates
collectors_to_remove = []
for collector in list(REGISTRY._collector_to_names.keys()):
    if 'prediction' in str(collector).lower():
        collectors_to_remove.append(collector)

for collector in collectors_to_remove:
    REGISTRY.unregister(collector)

# Create metrics
PREDICTION_COUNT = Counter('prediction_requests_total', 'Total prediction requests')
PREDICTION_LATENCY = Histogram('prediction_latency_seconds', 'Prediction latency')

print("✅ Prometheus metrics configured")

✅ Prometheus metrics configured
